# 🥈 Silver Layer — Clean and Validate Loan Transactions

**What we're doing:** Taking Bronze raw data and cleaning it.
- Remove failed/invalid transactions
- Standardize date formats
- Add useful columns for reporting

In [ ]:
from pyspark.sql import functions as F

# Read from Bronze
df = spark.table('bronze_loan_transactions')
print(f'Bronze records: {df.count()}')

In [ ]:
# Clean: keep only completed transactions, fix dates, add month column
df_silver = df \
    .filter(F.col('Status') == 'Completed') \
    .withColumn('TransactionDate', F.to_date('TransactionDate', 'yyyy-MM-dd')) \
    .withColumn('Month', F.date_format('TransactionDate', 'yyyy-MM')) \
    .withColumn('Amount', F.col('Amount').cast('double'))

print(f'✅ Silver records after cleaning: {df_silver.count()}')
df_silver.show(10)

In [ ]:
# Save Silver table
df_silver.write.format('delta').mode('overwrite').saveAsTable('silver_loan_transactions')
print('✅ Silver table saved!')

In [ ]:
%%sql
-- Transactions by type in Silver (clean data only)
SELECT TransactionType, COUNT(*) as Count, ROUND(SUM(Amount),2) as TotalAmount
FROM silver_loan_transactions
GROUP BY TransactionType